# 02 Species Prediction using DINOv2

In [ ]:
import os
import torch
import timm
import rasterio
import numpy as np
import pandas as pd
from torchvision import transforms
from tqdm import tqdm

## Configuration

In [ ]:
CROP_DIR = "outputs/cropped_crowns"
MODEL_PATH = "models/best_dinov2_linear.pth"
OUTPUT_CSV = "outputs/predictions.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 224

## Load trained model

In [ ]:
backbone = timm.create_model(
    "vit_base_patch14_dinov2",
    pretrained=False,
    num_classes=0,
    img_size=224
)

classifier = torch.nn.Linear(768, 2)

model = torch.nn.Sequential(backbone, classifier)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

## Image transform

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Normalize(
        mean=(0.485,0.456,0.406),
        std=(0.229,0.224,0.225)
    )
])

## Run prediction

In [ ]:
results = []

for img_name in tqdm(os.listdir(CROP_DIR)):

    img_path = os.path.join(CROP_DIR, img_name)

    with rasterio.open(img_path) as src:
        img = src.read([1,2,3]).transpose(1,2,0)

    img = np.clip(img,0,255).astype(np.uint8)

    input_tensor = transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(input_tensor)
        prob = torch.softmax(logits,dim=1)

    pred = torch.argmax(prob,dim=1).item()
    confidence = prob[0][pred].item()

    label = "acacia" if pred == 0 else "non_acacia"

    results.append({
        "image_name":img_name,
        "label":label,
        "confidence":confidence
    })

df_pred = pd.DataFrame(results)
df_pred.to_csv(OUTPUT_CSV,index=False)
df_pred.head()